# Notebook : Population genomics for *Laurus nobilis* & *Laurus azorica* & *Laurus novocanariensis*

## 1 : Merging the all sites VCF from *Laurus nobilis* & *Laurus azorica* & *Laurus novocanariensis*

### Script_merge_vcf.sh

In [ ]:
#!/bin/bash

#SBATCH --job-name=Merge_vcf
#SBATCH -p workq
#SBATCH --time=1-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

set -euo pipefail

#############################
# 1. MODULES
#############################

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

#############################
# 2. INPUT
#############################

VCF1="L_azorica_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
VCF2="L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
VCF3="L_novocanariensis_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"

GVCF1="${VCF1}.gz"
GVCF2="${VCF2}.gz"
GVCF3="${VCF3}.gz"

# Noms propres et uniques
MERGED="laurus_allsites.vcf.gz"
FILTERED="laurus_allsites_filtInd.vcf.gz"
FINAL="laurus_snps_final.vcf.gz"

#####################
# 3. GZIP & INDEX
#####################

bgzip -f $VCF1
bgzip -f $VCF2
bgzip -f $VCF3

bcftools index -t $GVCF1
bcftools index -t $GVCF2
bcftools index -t $GVCF3

###################
# 4. MERGE
###################

bcftools merge \
    --merge all \
    -Oz \
    -o $MERGED \
    $GVCF1 $GVCF2 $GVCF3

bcftools index -t $MERGED

echo "Merge terminé : $MERGED"

#####################
# 5. FILTER INDIVIDUALS
#####################

# Remove individuals with too much missing data

bcftools view \
    -s ^Lm-M02Cb,Lm-M03Cb,Lm-F01Cb \
    -Oz \
    -o $FILTERED \
    $MERGED

bcftools index -t $FILTERED

#############################
# 6. SNP FILTER
#############################

vcftools \
  --gzvcf "$FILTERED" \
  --remove-indels \
  --minDP 10 \
  --max-missing 0.9 \
  --min-alleles 2 \
  --max-alleles 2 \
  --minQ 30 \
  --recode \
  --recode-INFO-all \
  --out laurus_snps

# Compression propre
bgzip -f laurus_snps.recode.vcf
mv laurus_snps.recode.vcf.gz $FINAL

bcftools index -t $FINAL

## 2. Population structure: PCA & ADMIXTURE

### Script to filter the LD with plink and perform PCA & ADMIXTURE

In [ ]:
#!/bin/bash

#SBATCH --job-name=plink_admixture
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

# ======================
# MODULES
# ======================
module load bioinfo/PLINK/2.00a4
module load bioinfo/ADMIXTURE/1.3.0

# ======================
# PATH VARIABLES
# ======================
WORKDIR="/home/tbertrand/work/Laurus_pop_gen/PLINK"
INPUT_VCF="${WORKDIR}/laurus_snps_final.vcf.gz"

PRUNE_PREFIX="${WORKDIR}/laurus_snps_final"
LD_PREFIX="${WORKDIR}/laurus_snps_final_LD_prunned"

PCA_DIR="${WORKDIR}/PCA"
ADMIX_DIR="${WORKDIR}/ADMIXTURE"

# Create output directories
mkdir -p ${PCA_DIR}
mkdir -p ${ADMIX_DIR}

# ======================
# LD PRUNING
# ======================
plink2 --vcf ${INPUT_VCF} \
    --indep-pairwise 20 5 0.1 \
    --out ${PRUNE_PREFIX} \
    --set-missing-var-ids @:# \
    --allow-extra-chr 

# ======================
# CREATE BED FILE
# ======================
plink2 --vcf ${INPUT_VCF} \
    --extract ${PRUNE_PREFIX}.prune.in \
    --out ${LD_PREFIX} \
    --set-missing-var-ids @:# \
    --allow-extra-chr \
    --make-bed

# ======================
# PCA
# ======================
plink2 --bfile ${LD_PREFIX} \
    --pca 10 \
    --allow-extra-chr \
    --out ${PCA_DIR}/laurus_pca

# ======================
# ADMIXTURE PREP
# ======================
# Fix chromosome column
awk '{$1="0";print $0}' ${LD_PREFIX}.bim > ${LD_PREFIX}.bim.tmp
mv ${LD_PREFIX}.bim.tmp ${LD_PREFIX}.bim

# Move working files to ADMIXTURE dir
cp ${LD_PREFIX}.bed ${ADMIX_DIR}/
cp ${LD_PREFIX}.bim ${ADMIX_DIR}/
cp ${LD_PREFIX}.fam ${ADMIX_DIR}/

cd ${ADMIX_DIR}

# ======================
# ADMIXTURE RUN
# ======================
for K in {2..10}
do
    admixture --cv laurus_snps_final_LD_prunned.bed ${K} > log${K}.out
done

# ======================
# CV ERROR EXTRACTION
# ======================
grep "CV" log*.out | \
awk '{print $3,$4}' | \
sed -e 's/(//;s/)//;s/://;s/K=//' \
> cv_error.txt

echo "Pipeline completed successfully"

### Import results PCA & ADMIXTURE

In [ ]:
#PCA
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/PLINK/PCA /Users/cibio/Desktop/Laurus/Pop_gen/PLINK

#ADMIXTURE
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/PLINK/ADMIXTURE /Users/cibio/Desktop/Laurus/Pop_gen/PLINK

scp "tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/PLINK/laurus_snps_final_LD_prunned*" /Users/cibio/Desktop/Laurus/Pop_gen/PLINK/

### Convertion Laurus_snps_final_LD_prunned en VCF

In [ ]:
#!/bin/bash
#SBATCH --job-name=plink_admixture
#SBATCH -p workq

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16
module load bioinfo/PLINK/2.00a4

plink2 \
  --bfile laurus_snps_final_LD_prunned \
  --recode vcf \
  --out laurus_snps_final_LD_prunned

bcftools sort laurus_snps_final_LD_prunned.vcf -Oz -o laurus_snps_final_LD_prunned.sorted.vcf.gz

tabix -p vcf laurus_snps_final_LD_prunned.sorted.vcf.gz

## 3. SNP's Phylogeny with IQTREE

### Script to convert the SNP's data into phylip format and perform the phylogeny using IQtree GTR+ASC model

In [ ]:
scp https://github.com/edgardomortiz/vcf2phylip/blob/master/vcf2phylip.py

In [ ]:
#!/bin/bash

#SBATCH --job-name=plink_admixture
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

# ======================
# MODULES
# ======================
module load bioinfo/PLINK/2.00a4
module load bioinfo/ADMIXTURE/1.3.0
module load devel/Miniforge/Miniforge3
module load compilers/gcc/12.2.0
module load bioinfo/IQ-TREE/3.0.1

# ======================
# PATH VARIABLES
# ======================

WORKDIR="/home/tbertrand/work/Laurus_pop_gen/IQTREE"
VCF="/home/tbertrand/work/Laurus_pop_gen/PLINK/laurus_snps_final_LD_prunned.vcf.gz"

# ======================
# Convert VCF to PHYLIP
# ======================

python ${WORKDIR}/vcf2phylip.py -i $LD_SNP --output-folder $WORKDIR


# ======================
# IQTREE
# ======================

iqtree3 -s laurus_snps_final_LD_prunned.min4.phy -m GTR+ASC -T 4


In [ ]:
scp -r tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/IQTREE /Users/cibio/Desktop/Laurus/Pop_gen/PLINK

/Users/cibio/Desktop/Laurus/Pop_gen/PLINK
/home/tbertrand/work/Laurus_pop_gen/IQTREE

### Script to install, construct a custome refercne genome  database and annotate the SNP's

#### Install SnpEff

In [ ]:
#Need java >1.8.5
module load devel/java/24.0.2

module load bioinfo/SnpEff/5.4a

java -jar $SNPEFF_HOME/snpEff.jar -h
snpEff -h


#### Construct the custom reference genome database & annotate the SNP

In [ ]:
#!/bin/bash
#SBATCH --job-name=plink_admixture
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

module load devel/java/24.0.2
module load bioinfo/SnpEff/5.4a

# =========================
# SnpEff annotation script
# Laurus_azorica database
# =========================

# ---- INPUTS ----
VCF=/home/tbertrand/work/Laurus_pop_gen/TEST_MERGE/laurus_snps_final.vcf.gz
OUTDIR=/home/tbertrand/work/Laurus_pop_gen/SnpEff
GENOME="Laurus_azorica"

# ---- PATHS ----
SNPEFF_DIR=/home/tbertrand/work/Laurus_pop_gen/SnpEff/snpeff_custom
CONFIG=/home/tbertrand/work/Laurus_pop_gen/SnpEff/snpeff_custom/snpEff.config

# ---- OUTPUT ----

OUT_VCF=${OUTDIR}/laurus_snps_final.annotated.vcf
STATS_HTML=${OUTDIR}/laurus_snps_final.snpeff.html

echo "Annotating: ${VCF}"
echo "Output: ${OUT_VCF}"

# ---- RUN SNPEFF ----
snpEff -v -canon ${GENOME} \
    -c ${CONFIG} \
    -stats ${STATS_HTML} \
    ${VCF} > ${OUT_VCF}

echo "Done"

In [ ]:
scp "tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/SnpEff/laurus_snps_final.snpeff.html" /Users/cibio/Desktop/Laurus/Pop_gen

In [ ]:
#!/bin/bash
#SBATCH --job-name=plink_admixture
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

module load devel/java/24.0.2
module load bioinfo/SnpEff/5.4a
module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

OUTDIR=/home/tbertrand/work/Laurus_pop_gen/SnpEff
ANN_VCF=${OUTDIR}/laurus_snps_final.annotated.vcf

# =========================
# 2. SYNONYMOUS SNPs
# =========================
echo "Extracting synonymous SNPs..."

java -jar $SNPEFF_HOME/SnpSift.jar filter "ANN[*].EFFECT has 'synonymous_variant'" \
    ${ANN_VCF} | bgzip -c > ${OUTDIR}/laurus_synonymous.vcf.gz

tabix -p vcf ${OUTDIR}/laurus_synonymous.vcf.gz

# =========================
# 3. NON-SYNONYMOUS SNPs
# =========================
echo "Extracting non-synonymous SNPs..."

java -jar $SNPEFF_HOME/SnpSift.jar filter "ANN[*].EFFECT has 'missense_variant' || ANN[*].EFFECT has 'stop_gained' || ANN[*].EFFECT has 'start_lost' || ANN[*].EFFECT has 'frameshift_variant' || ANN[*].EFFECT has 'splice_region_variant' || ANN[*].EFFECT has 'splice_donor_variant' || ANN[*].EFFECT has 'splice_acceptor_variant'" \
    ${ANN_VCF} | bgzip -c > ${OUTDIR}/laurus_nonsynonymous.vcf.gz

tabix -p vcf ${OUTDIR}/laurus_nonsynonymous.vcf.gz


In [ ]:
#!/bin/bash
#SBATCH --job-name=plink_admixture
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

module load devel/java/24.0.2
module load bioinfo/SnpEff/5.4a
module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

OUTDIR=/home/tbertrand/work/Laurus_pop_gen/SnpEff
ANN_VCF=${OUTDIR}/laurus_snps_final.annotated.vcf

# =========================
# 2. CDS SNPs
# =========================

echo "Filtering protein-coding SNPs..."

java -jar $SNPEFF_HOME/SnpSift.jar filter "ANN[*].BIOTYPE = 'protein_coding'" \
    ${ANN_VCF} | bgzip -c > ${OUTDIR}/laurus_coding.vcf.gz

tabix -p vcf ${OUTDIR}/laurus_coding.vcf.gz


# =========================
# 2. SYNONYMOUS SNPs
# =========================

echo "Extracting synonymous SNPs..."

java -jar $SNPEFF_HOME/SnpSift.jar filter \
"(ANN[*].EFFECT has 'synonymous_variant')" \
${OUTDIR}/laurus_coding.vcf.gz | bgzip -c > ${OUTDIR}/laurus_synonymous.vcf.gz

tabix -p vcf ${OUTDIR}/laurus_synonymous.vcf.gz

# =========================
# 3. NON-SYNONYMOUS SNPs
# =========================

echo "Extracting non-synonymous SNPs..."

java -jar $SNPEFF_HOME/SnpSift.jar filter \
"(ANN[*].EFFECT has 'missense_variant' || ANN[*].EFFECT has 'stop_gained' || ANN[*].EFFECT has 'stop_lost' || ANN[*].EFFECT has 'start_lost')" \
${OUTDIR}/laurus_coding.vcf.gz | bgzip -c > ${OUTDIR}/laurus_nonsynonymous.vcf.gz

tabix -p vcf ${OUTDIR}/laurus_nonsynonymous.vcf.gz

#### Separate the SNP dataset in the different categories (SYN & NONSYN) and count the SNP per pop

In [ ]:
#!/bin/bash
#!/bin/bash
#SBATCH --job-name=VCF_split
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

VCF_SYN="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_synonymous.vcf.gz"
VCF_NON_SYN="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_nonsynonymous.vcf.gz"
VCF="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_snps_final.annotated.vcf.gz"

OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/pop_split"
mkdir -p "$OUTDIR"

########################
# POPULATIONS
########################

L_azorica="LaM16 LaM15 LaF10 LaF14 LaF9 LaF12 LaF8 LaF11 LaF13 LaF5 LaM1 LaF6 LaF2 LaF7 LaF4 LaM3 LaF1 LaM9 LaF15 LaM4 LaM6 LaM17 LaM5 LaM7 LaF3 LaM8 LaM10 LaM11 LaM12 LaF16 LaM13 LaM14"

L_nobilis_POR="Lm-M14Cb Lm-M11Cb Lm-M09b Lm-F16b Lm-F15Cbf Lm-F13Cbf Lm-F12Cf Lm-F09Cf Lm-F08Cbf Lm-M15Cb Lm-F07b Lm-F05Cbf Lm-M04b Lm-F02Cb Lm-M05b Lm-M13Cf Lm-M12Cb Lm-M07Cf Lm-F04Cbf Lm-F06Cbf Lm-M10b Lm-F10Cb Lm-F03Cb Lm-M08Cb Lm-F14Cbf Lm-M06Cb Lm-M01Cf"

L_nobilis_ITA="OR05F OR03M OR09M OR01M OR04Mb OR05M OR01F OR03Fb OR04F OR06Fb OR10M OR09F OR02M OR07M OR08F"

L_nobilis_GRE="GRE6-7_1 GRE5_2b GRE3-4_1b GRE3-4_2b GRE1-2_5b"

L_nobilis_TUN="TUNI06-M TUNI10-Mb TUNI13-Fb TUNI14-Fb TUNI11-Fb"

L_novocanareinsis="LN-T28_F LN-T27_F LN-T22_M LN-T20_M LN-T18b_M LN-T17b_M LN-T23_M LN-T16_M LN-T26_F LN-T24_M LN-T1_M LN-T2_F LN-T3_F LN-T14_M LN-T4_M LN-T11_M LN-T21_M LN-T5_F LN-T6_M LN-T7_M LN-T25 LN-T19_M LN-T10_M LN-T15_M LN-T12_M LN-T13_F"

########################
# SYNONYMOUS
########################

echo "=== SYNONYMOUS ==="

echo $L_azorica | tr ' ' '\n' > "$OUTDIR/La.samples.txt"
bcftools view -S "$OUTDIR/La.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.La.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.La.vcf.gz"
bcftools view -H "$OUTDIR/syn.La.vcf.gz" | wc -l > "$OUTDIR/count_syn.txt"


echo $L_nobilis_POR | tr ' ' '\n' > "$OUTDIR/Lm.samples.txt"
bcftools view -S "$OUTDIR/Lm.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.Lm.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.Lm.vcf.gz"
bcftools view -H "$OUTDIR/syn.Lm.vcf.gz" | wc -l >> "$OUTDIR/count_syn.txt"

echo $L_nobilis_ITA | tr ' ' '\n' > "$OUTDIR/OR.samples.txt"
bcftools view -S "$OUTDIR/OR.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.OR.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.OR.vcf.gz"
bcftools view -H "$OUTDIR/syn.OR.vcf.gz" | wc -l >> "$OUTDIR/count_syn.txt"

echo $L_nobilis_GRE | tr ' ' '\n' > "$OUTDIR/GRE.samples.txt"
bcftools view -S "$OUTDIR/GRE.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.GRE.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.GRE.vcf.gz"
bcftools view -H "$OUTDIR/syn.GRE.vcf.gz" | wc -l >> "$OUTDIR/count_syn.txt"

echo $L_nobilis_TUN | tr ' ' '\n' > "$OUTDIR/TUNI.samples.txt"
bcftools view -S "$OUTDIR/TUNI.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.TUNI.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.TUNI.vcf.gz"
bcftools view -H "$OUTDIR/syn.TUNI.vcf.gz" | wc -l >> "$OUTDIR/count_syn.txt"

echo $L_novocanareinsis | tr ' ' '\n' > "$OUTDIR/LN.samples.txt"
bcftools view -S "$OUTDIR/LN.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/syn.LN.vcf.gz" "$VCF_SYN"
tabix -f "$OUTDIR/syn.LN.vcf.gz"
bcftools view -H "$OUTDIR/syn.LN.vcf.gz" | wc -l >> "$OUTDIR/count_syn.txt"


########################
# NON SYNONYMOUS
########################

echo "=== NON SYNONYMOUS ==="

bcftools view -S "$OUTDIR/La.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.La.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.La.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.La.vcf.gz" | wc -l > "$OUTDIR/count_nonsyn.txt"

bcftools view -S "$OUTDIR/Lm.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.Lm.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.Lm.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.Lm.vcf.gz" | wc -l >> "$OUTDIR/count_nonsyn.txt"

bcftools view -S "$OUTDIR/OR.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.OR.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.OR.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.OR.vcf.gz" | wc -l >> "$OUTDIR/count_nonsyn.txt"

bcftools view -S "$OUTDIR/GRE.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.GRE.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.GRE.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.GRE.vcf.gz" | wc -l >> "$OUTDIR/count_nonsyn.txt"

bcftools view -S "$OUTDIR/TUNI.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.TUNI.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.TUNI.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.TUNI.vcf.gz" | wc -l >> "$OUTDIR/count_nonsyn.txt"

bcftools view -S "$OUTDIR/LN.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/nonsyn.LN.vcf.gz" "$VCF_NON_SYN"
tabix -f "$OUTDIR/nonsyn.LN.vcf.gz"
bcftools view -H "$OUTDIR/nonsyn.LN.vcf.gz" | wc -l >> "$OUTDIR/count_nonsyn.txt"


########################
# GLOBAL VCF
########################

echo "=== GLOBAL VCF ==="

bcftools view -S "$OUTDIR/La.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.La.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.La.vcf.gz"
bcftools view -H "$OUTDIR/global.La.vcf.gz" | wc -l > "$OUTDIR/count_global.txt"

bcftools view -S "$OUTDIR/Lm.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.Lm.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.Lm.vcf.gz"
bcftools view -H "$OUTDIR/global.Lm.vcf.gz" | wc -l >> "$OUTDIR/count_global.txt"

bcftools view -S "$OUTDIR/OR.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.OR.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.OR.vcf.gz"
bcftools view -H "$OUTDIR/global.OR.vcf.gz" | wc -l >> "$OUTDIR/count_global.txt"

bcftools view -S "$OUTDIR/GRE.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.GRE.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.GRE.vcf.gz"
bcftools view -H "$OUTDIR/global.GRE.vcf.gz" | wc -l >> "$OUTDIR/count_global.txt"

bcftools view -S "$OUTDIR/TUNI.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.TUNI.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.TUNI.vcf.gz"
bcftools view -H "$OUTDIR/global.TUNI.vcf.gz" | wc -l >> "$OUTDIR/count_global.txt"

bcftools view -S "$OUTDIR/LN.samples.txt" -c 1 -v snps -Oz -o "$OUTDIR/global.LN.vcf.gz" "$VCF"
tabix -f "$OUTDIR/global.LN.vcf.gz"
bcftools view -H "$OUTDIR/global.LN.vcf.gz" | wc -l >> "$OUTDIR/count_global.txt"


echo "DONE"

#### Compute the pi per site in the different dataset using VCFtools

In [ ]:
#!/bin/bash

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

############################
# INPUTS
############################

VCF_SYN="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_synonymous.vcf.gz"
VCF_NON_SYN="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_nonsynonymous.vcf.gz"
VCF="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/VCF/laurus_snps_final.annotated.vcf.gz"

POP="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/pixy/population.txt"

OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/pi_vcftools"
mkdir -p "$OUTDIR"

############################
# FUNCTION
############################

run_pi () {
    VCF=$1
    NAME=$2

    echo "========================"
    echo "Processing $NAME"
    echo "========================"

    vcftools \
        --gzvcf $VCF \
        --site-pi \
        --keep $POP \
        --out $OUTDIR/$NAME

    echo "Done $NAME"
}

############################
# RUN
############################

run_pi $VCF_SYN "synonymous"
run_pi $VCF_NON_SYN "nonsynonymous"
run_pi $VCF "global"

echo "All done"

In [ ]:
scp -r "tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Pi/pi_vcftools" /Users/cibio/Desktop/Laurus/Pop_gen/Diversity

#### Computing the Callable site per population

In [ ]:
#!/bin/bash
#SBATCH --job-name=callable_sites
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

############################
# INPUT
############################

VCF_ALL="/home/tbertrand/work/Laurus_pop_gen/TEST_MERGE/laurus_allsites.vcf.gz"

POP_DIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Pi"

POPS=(
"L_azorica"
"L_nobilis_POR"
"L_nobilis_ITA"
"L_nobilis_GRE"
"L_nobilis_TUN"
"L_novocanariensis"
)

OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Callable_site_counts"
mkdir -p "$OUTDIR"

############################
# STEP 1: GLOBAL FILTER
############################

echo "Filtering global VCF (compressed)..."

vcftools \
    --gzvcf $VCF_ALL \
    --minDP 10 \
    --max-missing 0.9 \
    --recode \
    --recode-INFO-all \
    --stdout | bgzip -c > ${OUTDIR}/filtered_all.vcf.gz

tabix -p vcf ${OUTDIR}/filtered_all.vcf.gz

FILTERED_VCF="${OUTDIR}/filtered_all.vcf.gz"

############################
# STEP 2: COUNT PER POP
############################

> ${OUTDIR}/sites_per_population.txt

for POP in "${POPS[@]}"; do

    echo "=========================="
    echo "Processing $POP"
    echo "=========================="

    N_SITES=$(bcftools view \
        -S ${POP_DIR}/${POP}.txt \
        -H $FILTERED_VCF | wc -l)

    echo -e "${POP}\t${N_SITES}" >> ${OUTDIR}/sites_per_population.txt

done

echo "DONE"

In [ ]:
#!/bin/bash
#SBATCH --job-name=callable_sites
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=1

module load bioinfo/VCFtools/0.1.16

POP_DIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Pi"

POPS=(
"L_azorica"
"L_nobilis_POR"
"L_nobilis_ITA"
"L_nobilis_GRE"
"L_nobilis_TUN"
"L_novocanariensis"
)

OUTDIR="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/Callable_site_counts"

FILTERED_VCF="${OUTDIR}/filtered_all.vcf.gz"

> ${OUTDIR}/sites_per_population.txt

for POP in "${POPS[@]}"; do

    echo "=========================="
    echo "Processing $POP"
    echo "=========================="

    N_SITES=$(vcftools \
        --gzvcf $FILTERED_VCF \
        --keep ${POP_DIR}/${POP}.txt \
        --max-missing 0.001 \
        --remove-indels \
        --recode \
        --stdout \
    | grep -v "^#" \
    | wc -l)

    echo -e "${POP}\t${N_SITES}" >> ${OUTDIR}/sites_per_population.txt

done

## 4. Diversity and divergence analysis with Pixy using the all site vcf

Compute Pi | Fst | Dxy

In [ ]:
#!/bin/bash
#!/bin/bash
#SBATCH --job-name=VCF_split
#SBATCH -p workq
#SBATCH --time=4:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4


module load devel/Miniconda/Miniconda3
module load bioinfo/Bcftools/1.17
module load bioinfo/pixy/1.2.11


VCF="/home/tbertrand/work/Laurus_pop_gen/TEST_MERGE/laurus_allsites.vcf.gz"
POP="/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/pixy/population.txt"

pixy --stats pi fst dxy \
--vcf $VCF \
--populations $POP \
--window_size 10000 \
--n_cores 4


In [ ]:
scp -r "tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/Laurus_pop_gen/DIVERSITY/pixy" /Users/cibio/Desktop/Laurus/Pop_gen/Diversity

## Annexe Synonymous VS nonSynonymous Diversity 

See the PiNPiS.ipynb notebook

## Annexe SNP annotation using SnpEff & Diversity analysis with VCFtools